
# AIA 2025–2026 HARP-Block v3 Sharded Miner — VM Ready

This notebook replaces the slow **six JSOC exports per individual sample** strategy.

## Core idea

Instead of:

```text
1 sample × 6 wavelengths = 6 JSOC export jobs
```

the miner groups required timestamps by **HARPNUM** and **24-hour blocks**:

```text
1 HARP/time block × 6 wavelength-sequence exports
→ many model-ready samples
```

Each wavelength request returns a tracked time series of active-region cutouts. The notebook then:

1. matches each returned AIA image to the required SHARP timestamp;
2. locally extracts the target-specific crop using the FITS WCS;
3. resizes it to `512 × 512`;
4. applies the same historical preprocessing used for 2010–2024;
5. stacks the six channels;
6. uploads each `.npz` immediately to Google Cloud Storage;
7. checkpoints sample and block progress for safe restart.

## Safety

The notebook defaults to `BLOCK_CANARY` mode. It must pass a small block test before `PRODUCTION` mode is enabled.

Official JSOC/DRMS behaviour used here:

- query form: `Series[timespan@cadence][wavelength]{image}`;
- `im_patch` server-side cutouts;
- `t=0` enables solar-rotation tracking;
- one pending export at a time per registered email;
- RequestIDs are saved and reopened after interruption.

## VM execution

Run this notebook on the prepared Compute Engine VM inside `tmux`.

For 2025:

```bash
export TARGET_YEAR=2025
export JSOC_EMAIL=abmoses2000@gmail.com
export WORKER_ID=aia2025
export RUN_MODE=BLOCK_CANARY
```

For 2026, after the 2025 block canary succeeds:

```bash
export TARGET_YEAR=2026
export JSOC_EMAIL=worky4work@gmail.com
export WORKER_ID=aia2026
export RUN_MODE=BLOCK_CANARY
```

After QA passes, change `RUN_MODE=PRODUCTION`.


> **Production sharding:** set `NUM_SHARDS` and `SHARD_INDEX` to split the deterministic block plan into non-overlapping workers. The default `NUM_SHARDS=1` preserves single-worker behaviour.


In [1]:

# The VM environment already contains most packages.
# This cell is safe to rerun and installs only missing dependencies.

%pip install -q --upgrade \
    "drms>=0.9.1" \
    "astropy>=7.0" \
    "sunpy[map]>=7.0" \
    "scikit-image>=0.25" \
    "google-cloud-storage>=3.0"


Note: you may need to restart the kernel to use updated packages.


In [2]:

import os
import re
import gc
import sys
import json
import time
import math
import shutil
import hashlib
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import drms
from drms.exceptions import DrmsExportError
from astropy.io import fits
from astropy import units as u
from astropy.coordinates import SkyCoord
from skimage.transform import resize
from skimage.metrics import structural_similarity

import sunpy.map

print("Python:", sys.version)
print("DRMS:", drms.__version__)
print("SunPy:", sunpy.__version__)


Python: 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
DRMS: 0.9.1
SunPy: 7.1.2


/home/abmoses2000/solar_flare_aia/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

In [3]:

# ============================================================
# ENVIRONMENT-AWARE CONFIGURATION
# ============================================================

PROJECT_ID = "sonorous-shore-450510-i4"
GCP_BUCKET = "gs://suryabench-sharp-pipeline-bamidele"

TARGET_YEAR = int(os.environ.get("TARGET_YEAR", "2025"))
JSOC_EMAIL = os.environ.get(
    "JSOC_EMAIL",
    "abmoses2000@gmail.com" if TARGET_YEAR == 2025 else "worky4work@gmail.com",
)
WORKER_ID = os.environ.get("WORKER_ID", f"aia{TARGET_YEAR}")
RUN_MODE = os.environ.get("RUN_MODE", "BLOCK_CANARY").upper()

# Deterministic, non-overlapping production sharding.
# Defaults preserve the original single-worker behaviour.
NUM_SHARDS = int(os.environ.get("NUM_SHARDS", "1"))
SHARD_INDEX = int(os.environ.get("SHARD_INDEX", "0"))

if RUN_MODE not in {"BLOCK_CANARY", "PRODUCTION"}:
    raise ValueError("RUN_MODE must be BLOCK_CANARY or PRODUCTION.")

if NUM_SHARDS < 1:
    raise ValueError("NUM_SHARDS must be at least 1.")

if not 0 <= SHARD_INDEX < NUM_SHARDS:
    raise ValueError(
        f"SHARD_INDEX must be in [0, {NUM_SHARDS - 1}], "
        f"received {SHARD_INDEX}."
    )

if RUN_MODE == "BLOCK_CANARY" and NUM_SHARDS != 1:
    raise ValueError(
        "BLOCK_CANARY must run with NUM_SHARDS=1. "
        "Use sharding only in PRODUCTION mode."
    )

AIA_WAVELENGTHS = [94, 131, 171, 193, 211, 335]
IMAGE_SIZE = 512

# Time grouping
BLOCK_HOURS = 24
TARGET_CADENCE_MIN = 96
MAX_TARGET_TIME_DIFFERENCE_SEC = 180
MAX_GAP_WITHIN_TRACK_SEC = 3 * 3600

# The server-side tracked patch is deliberately larger than the
# target-specific crop. Each target is then cropped locally using WCS.
BLOCK_PATCH_MARGIN_ARCSEC = 160.0
MIN_BLOCK_PATCH_ARCSEC = 300.0
MAX_BLOCK_PATCH_ARCSEC = 1100.0

# Historical geometry constants retained for compatibility with the pilot.
FULL_DISK_SIZE = 4096
IMAGE_CENTER = FULL_DISK_SIZE // 2
AIA_PIXEL_SCALE_ARCSEC = 0.6
SOLAR_RADIUS_ARCSEC = 976.0
SOLAR_RADIUS_PIX = SOLAR_RADIUS_ARCSEC / AIA_PIXEL_SCALE_ARCSEC
CROP_SCALE = 1.2
CROP_PADDING_PIX = 30
MIN_CROP_PIX = 64

# JSOC queue protection
JSOC_MAX_RETRIES = 10
JSOC_INITIAL_BACKOFF_SEC = 20
JSOC_MAX_BACKOFF_SEC = 300
JSOC_WAIT_TIMEOUT_SEC = 7200
JSOC_COOLDOWN_SEC = 12

# Runtime limits
MAX_BLOCKS_THIS_RUN = (
    int(os.environ["MAX_BLOCKS_THIS_RUN"])
    if os.environ.get("MAX_BLOCKS_THIS_RUN")
    else (1 if RUN_MODE == "BLOCK_CANARY" else None)
)
MIN_FREE_DISK_GB = 15

# The canary block is chosen around a previously successful individual sample.
CANARY_SAMPLE_IDS = {
    2025: [
        "20250602_1348_HARP13299_NOAA14100",
        "20250628_2248_HARP13424_NOAA14122",
    ],
    2026: [
        "20260210_0400_HARP14361_NOAA14370",
        "20260211_1648_HARP14371_NOAA14373",
    ],
}

BASE = Path.home() / "solar_flare_aia"
LOCAL_ROOT = BASE / "harp_block_miner" / f"{RUN_MODE.lower()}_{WORKER_ID}"
LOCAL_META = LOCAL_ROOT / "metadata"
LOCAL_TEMP = LOCAL_ROOT / "temp_blocks"
LOCAL_OUTPUT = LOCAL_ROOT / "samples_npz"
LOCAL_LOG = LOCAL_META / f"sample_log_{WORKER_ID}.csv"
LOCAL_BLOCK_LOG = LOCAL_META / f"block_log_{WORKER_ID}.csv"
LOCAL_BLOCK_PLAN = LOCAL_META / f"block_plan_{WORKER_ID}.csv"

for directory in [LOCAL_ROOT, LOCAL_META, LOCAL_TEMP, LOCAL_OUTPUT]:
    directory.mkdir(parents=True, exist_ok=True)

if RUN_MODE == "BLOCK_CANARY":
    GCP_RUN_ROOT = f"{GCP_BUCKET}/jsoc_harp_block_canary_v1/{WORKER_ID}"
else:
    GCP_RUN_ROOT = f"{GCP_BUCKET}/jsoc_2025_2026_production_v1"

GCP_OUTPUT_ROOT = f"{GCP_RUN_ROOT}/samples_npz/{TARGET_YEAR}"
GCP_WORKER_META = f"{GCP_RUN_ROOT}/metadata/workers/{WORKER_ID}"

GCP_METADATA_CANDIDATES = [
    f"{GCP_BUCKET}/metadata/curated_sharp_suryabench_true96min_48h_AR_SPECIFIC_2010_2026.csv",
    f"{GCP_BUCKET}/metadata/curated_2025_2026_AR_SPECIFIC_EXTENSION.csv",
]

PILOT_GCP_ROOT = (
    f"{GCP_BUCKET}/jsoc_2025_2026_pilot/samples_npz/{TARGET_YEAR}"
)

print("=" * 80)
print("TARGET_YEAR:", TARGET_YEAR)
print("JSOC_EMAIL:", JSOC_EMAIL)
print("WORKER_ID:", WORKER_ID)
print("RUN_MODE:", RUN_MODE)
print("NUM_SHARDS:", NUM_SHARDS)
print("SHARD_INDEX:", SHARD_INDEX)
print("MAX_BLOCKS_THIS_RUN:", MAX_BLOCKS_THIS_RUN)
print("LOCAL_ROOT:", LOCAL_ROOT)
print("GCP_OUTPUT_ROOT:", GCP_OUTPUT_ROOT)
print("=" * 80)


TARGET_YEAR: 2025
JSOC_EMAIL: sleekwebdesigner@gmail.com
WORKER_ID: aia2025-s1
RUN_MODE: PRODUCTION
NUM_SHARDS: 2
SHARD_INDEX: 1
MAX_BLOCKS_THIS_RUN: None
LOCAL_ROOT: /home/abmoses2000/solar_flare_aia/harp_block_miner/production_aia2025-s1
GCP_OUTPUT_ROOT: gs://suryabench-sharp-pipeline-bamidele/jsoc_2025_2026_production_v1/samples_npz/2025


## 2. Cloud and JSOC preflight

In [4]:

def run_command(command, check=True, capture=True):
    result = subprocess.run(
        command,
        text=True,
        capture_output=capture,
    )
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {' '.join(command)}\n"
            f"{result.stderr[-3000:] if result.stderr else ''}"
        )
    return result


def gcp_exists(path):
    return run_command(
        ["gcloud", "storage", "ls", path],
        check=False,
    ).returncode == 0


print("Bucket access:")
bucket_test = run_command(
    ["gcloud", "storage", "ls", GCP_BUCKET],
    check=True,
)
print(bucket_test.stdout[:1000])
print("✅ Bucket access works.")

jsoc_public = drms.Client()
registered = jsoc_public.check_email(JSOC_EMAIL)
print("JSOC registered:", registered, "|", JSOC_EMAIL)
if not registered:
    raise RuntimeError(f"JSOC email is not registered: {JSOC_EMAIL}")

jsoc = drms.Client(email=JSOC_EMAIL)
assert jsoc.email == JSOC_EMAIL
print("✅ JSOC client is using the intended email.")


Bucket access:


gs://suryabench-sharp-pipeline-bamidele/baseline_results/
gs://suryabench-sharp-pipeline-bamidele/jsoc_2025_2026_canary/
gs://suryabench-sharp-pipeline-bamidele/jsoc_2025_2026_pilot/
gs://suryabench-sharp-pipeline-bamidele/jsoc_2025_2026_production_v1/
gs://suryabench-sharp-pipeline-bamidele/jsoc_2026_canary_parallel/
gs://suryabench-sharp-pipeline-bamidele/jsoc_harp_block_canary_v1/
gs://suryabench-sharp-pipeline-bamidele/manifests/
gs://suryabench-sharp-pipeline-bamidele/metadata/
gs://suryabench-sharp-pipeline-bamidele/samples_npz/

✅ Bucket access works.


JSOC registered: True | sleekwebdesigner@gmail.com


✅ JSOC client is using the intended email.


## 3. Load and validate corrected AR-specific metadata

In [5]:

def copy_first_existing(candidates, destination):
    for candidate in candidates:
        print("Checking:", candidate)
        if not gcp_exists(candidate):
            continue
        run_command(
            ["gcloud", "storage", "cp", candidate, str(destination)],
            check=True,
        )
        if destination.exists() and destination.stat().st_size > 0:
            print("✅ Copied:", candidate)
            return candidate
    raise FileNotFoundError("No compatible corrected metadata file was found.")


metadata_path = LOCAL_META / "corrected_ar_specific_metadata.csv"
metadata_source = copy_first_existing(
    GCP_METADATA_CANDIDATES,
    metadata_path,
)

raw_df = pd.read_csv(metadata_path, low_memory=False)
print("Raw metadata:", raw_df.shape)


Checking: gs://suryabench-sharp-pipeline-bamidele/metadata/curated_sharp_suryabench_true96min_48h_AR_SPECIFIC_2010_2026.csv


✅ Copied: gs://suryabench-sharp-pipeline-bamidele/metadata/curated_sharp_suryabench_true96min_48h_AR_SPECIFIC_2010_2026.csv


Raw metadata: (141644, 50)


In [6]:

def clean_noaa(value):
    if pd.isna(value):
        return np.nan
    try:
        number = int(float(value))
        return number if number > 0 else np.nan
    except Exception:
        matches = re.findall(r"\d+", str(value))
        return int(matches[0]) if matches else np.nan


def prepare_metadata(frame):
    frame = frame.copy()

    frame["T_REC_dt"] = pd.to_datetime(
        frame["T_REC_dt"],
        errors="coerce",
    )

    if "NOAA_AR_clean" not in frame.columns:
        source = "NOAA_ARS" if "NOAA_ARS" in frame.columns else "NOAA_AR"
        frame["NOAA_AR_clean"] = frame[source].apply(clean_noaa)

    label_source = next(
        (
            column
            for column in [
                "label_48h_final",
                "label_48h_ar_specific",
                "label_48h",
            ]
            if column in frame.columns
        ),
        None,
    )
    if label_source is None:
        raise ValueError("No AR-specific 48-hour label column exists.")

    frame["label_48h_final"] = pd.to_numeric(
        frame[label_source],
        errors="coerce",
    )
    frame["HARPNUM"] = pd.to_numeric(frame["HARPNUM"], errors="coerce")
    frame["NOAA_AR_clean"] = pd.to_numeric(
        frame["NOAA_AR_clean"],
        errors="coerce",
    )

    required = [
        "T_REC_dt", "HARPNUM", "NOAA_AR_clean", "label_48h_final",
        "LON_MIN", "LON_MAX", "LAT_MIN", "LAT_MAX",
    ]
    missing = [column for column in required if column not in frame.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    frame = frame.dropna(subset=required).copy()
    frame["HARPNUM"] = frame["HARPNUM"].astype(int)
    frame["NOAA_AR_clean"] = frame["NOAA_AR_clean"].astype(int)
    frame["label_48h_final"] = frame["label_48h_final"].astype(int)

    if "sample_id" not in frame.columns:
        frame["sample_id"] = frame.apply(
            lambda row: (
                f"{row['T_REC_dt'].strftime('%Y%m%d_%H%M')}"
                f"_HARP{row['HARPNUM']}"
                f"_NOAA{row['NOAA_AR_clean']}"
            ),
            axis=1,
        )

    frame["year"] = frame["T_REC_dt"].dt.year
    frame = frame[frame["year"] == TARGET_YEAR].copy()

    # 2026 rows in the source file were already created using a safe
    # complete-future-window cutoff. Preserve that curated selection.
    frame = (
        frame.drop_duplicates("sample_id")
        .sort_values(["HARPNUM", "T_REC_dt"])
        .reset_index(drop=True)
    )

    return frame


df = prepare_metadata(raw_df)

print("Prepared rows:", len(df))
print(df["label_48h_final"].value_counts().sort_index())
print("Unique HARPs:", df["HARPNUM"].nunique())

expected_rows = 14774 if TARGET_YEAR == 2025 else 3201
if len(df) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} curated rows for {TARGET_YEAR}, "
        f"but found {len(df)}."
    )

print("✅ Metadata count matches the curated year total.")


Prepared rows: 14774
label_48h_final
0    13954
1      820
Name: count, dtype: int64
Unique HARPs: 261
✅ Metadata count matches the curated year total.


## 4. Geometry and preprocessing

In [7]:

def lonlat_to_pixel(lon_deg, lat_deg):
    lon = np.deg2rad(float(lon_deg))
    lat = np.deg2rad(float(lat_deg))
    x = IMAGE_CENTER + SOLAR_RADIUS_PIX * np.cos(lat) * np.sin(lon)
    y = IMAGE_CENTER - SOLAR_RADIUS_PIX * np.sin(lat)
    return float(x), float(y)


def target_geometry(row):
    corners = [
        (row["LON_MIN"], row["LAT_MIN"]),
        (row["LON_MIN"], row["LAT_MAX"]),
        (row["LON_MAX"], row["LAT_MIN"]),
        (row["LON_MAX"], row["LAT_MAX"]),
    ]
    pixels = [lonlat_to_pixel(lon, lat) for lon, lat in corners]
    xs = [item[0] for item in pixels]
    ys = [item[1] for item in pixels]

    center_x_pix = (min(xs) + max(xs)) / 2.0
    center_y_pix = (min(ys) + max(ys)) / 2.0

    width_pix = max(max(xs) - min(xs), MIN_CROP_PIX)
    height_pix = max(max(ys) - min(ys), MIN_CROP_PIX)
    crop_pix = max(width_pix, height_pix) * CROP_SCALE + CROP_PADDING_PIX

    return {
        "x_arcsec": (center_x_pix - IMAGE_CENTER) * AIA_PIXEL_SCALE_ARCSEC,
        "y_arcsec": (IMAGE_CENTER - center_y_pix) * AIA_PIXEL_SCALE_ARCSEC,
        "box_arcsec": crop_pix * AIA_PIXEL_SCALE_ARCSEC,
    }


def historical_preprocess(image):
    image = np.asarray(image, dtype=np.float32)
    image = np.nan_to_num(image, nan=0.0, posinf=0.0, neginf=0.0)
    image = np.clip(image, 0, None)
    image = np.log1p(image)

    low = float(image.min())
    high = float(image.max())
    if high <= low:
        return np.zeros_like(image, dtype=np.float32)

    return ((image - low) / (high - low)).astype(np.float32)


def read_map(path):
    solar_map = sunpy.map.Map(path)
    data = np.asarray(solar_map.data, dtype=np.float32)
    return solar_map, data


def crop_target_from_block(fits_path, row):
    solar_map, data = read_map(fits_path)
    geometry = target_geometry(row)

    coordinate = SkyCoord(
        geometry["x_arcsec"] * u.arcsec,
        geometry["y_arcsec"] * u.arcsec,
        frame=solar_map.coordinate_frame,
    )
    pixel = solar_map.world_to_pixel(coordinate)
    center_x = float(pixel.x.value)
    center_y = float(pixel.y.value)

    scale_x = abs(float(solar_map.scale.axis1.to_value(u.arcsec / u.pix)))
    scale_y = abs(float(solar_map.scale.axis2.to_value(u.arcsec / u.pix)))
    half_width = geometry["box_arcsec"] / (2.0 * scale_x)
    half_height = geometry["box_arcsec"] / (2.0 * scale_y)

    x0 = int(math.floor(center_x - half_width))
    x1 = int(math.ceil(center_x + half_width))
    y0 = int(math.floor(center_y - half_height))
    y1 = int(math.ceil(center_y + half_height))

    if x0 < 0 or y0 < 0 or x1 > data.shape[1] or y1 > data.shape[0]:
        raise ValueError(
            f"Target crop leaves block patch: "
            f"bounds={(x0, x1, y0, y1)}, shape={data.shape}"
        )

    crop = data[y0:y1, x0:x1]
    if crop.size == 0:
        raise ValueError("Empty local crop.")

    resized = resize(
        crop,
        (IMAGE_SIZE, IMAGE_SIZE),
        anti_aliasing=True,
        preserve_range=True,
    )
    return historical_preprocess(resized), {
        "block_shape": list(data.shape),
        "local_bounds": [x0, x1, y0, y1],
        "target_geometry": geometry,
    }


print("✅ Geometry and preprocessing functions ready.")


✅ Geometry and preprocessing functions ready.


## 5. Create HARP/time blocks

In [8]:

def make_blocks(frame, block_hours=24):
    blocks = []

    for harpnum, group in frame.groupby("HARPNUM"):
        group = group.sort_values("T_REC_dt").copy()
        current_indices = []
        block_start = None
        previous_time = None

        for index, row in group.iterrows():
            timestamp = pd.Timestamp(row["T_REC_dt"])

            must_split = False
            if block_start is not None:
                elapsed_hours = (
                    timestamp - block_start
                ).total_seconds() / 3600.0

                gap_seconds = (
                    timestamp - previous_time
                ).total_seconds()

                must_split = (
                    elapsed_hours >= block_hours
                    or gap_seconds > MAX_GAP_WITHIN_TRACK_SEC
                )

            if must_split and current_indices:
                blocks.append(group.loc[current_indices].copy())
                current_indices = []
                block_start = None

            if block_start is None:
                block_start = timestamp

            current_indices.append(index)
            previous_time = timestamp

        if current_indices:
            blocks.append(group.loc[current_indices].copy())

    plan_rows = []
    block_frames = {}

    for number, block in enumerate(blocks):
        first = block["T_REC_dt"].min()
        last = block["T_REC_dt"].max()
        harpnum = int(block["HARPNUM"].iloc[0])
        block_id = (
            f"{TARGET_YEAR}_HARP{harpnum}_"
            f"{first.strftime('%Y%m%d_%H%M')}_"
            f"{last.strftime('%Y%m%d_%H%M')}"
        )

        block_frames[block_id] = block.reset_index(drop=True)
        plan_rows.append(
            {
                "block_id": block_id,
                "HARPNUM": harpnum,
                "start": first,
                "end": last,
                "n_targets": len(block),
                "n_positive": int(block["label_48h_final"].sum()),
            }
        )

    return pd.DataFrame(plan_rows), block_frames


block_plan, block_frames = make_blocks(df, BLOCK_HOURS)

if RUN_MODE == "BLOCK_CANARY":
    wanted_ids = set(CANARY_SAMPLE_IDS[TARGET_YEAR])
    canary_block_ids = []

    for block_id, block in block_frames.items():
        if set(block["sample_id"]).intersection(wanted_ids):
            canary_block_ids.append(block_id)

    if not canary_block_ids:
        raise RuntimeError("No block contains the configured canary samples.")

    block_plan = block_plan[
        block_plan["block_id"].isin(canary_block_ids)
    ].copy()

block_plan = block_plan.sort_values(
    ["start", "HARPNUM", "block_id"]
).reset_index(drop=True)

# Preserve a stable global block index before selecting a shard.
block_plan["global_block_index"] = np.arange(len(block_plan), dtype=int)

if RUN_MODE == "PRODUCTION" and NUM_SHARDS > 1:
    total_blocks_before_sharding = len(block_plan)
    total_targets_before_sharding = int(block_plan["n_targets"].sum())

    block_plan = block_plan[
        block_plan["global_block_index"] % NUM_SHARDS == SHARD_INDEX
    ].copy().reset_index(drop=True)

    print(
        f"Shard {SHARD_INDEX}/{NUM_SHARDS - 1}: selected "
        f"{len(block_plan)} of {total_blocks_before_sharding} blocks."
    )
    print(
        "Targets in selected shard:",
        int(block_plan["n_targets"].sum()),
        "of",
        total_targets_before_sharding,
    )
else:
    print("Sharding disabled: using the complete selected block plan.")

block_plan["num_shards"] = NUM_SHARDS
block_plan["shard_index"] = SHARD_INDEX

block_plan.to_csv(LOCAL_BLOCK_PLAN, index=False)
run_command(
    [
        "gcloud", "storage", "cp",
        str(LOCAL_BLOCK_PLAN),
        f"{GCP_WORKER_META}/{LOCAL_BLOCK_PLAN.name}",
    ],
    check=True,
)

print("Blocks selected:", len(block_plan))
print("Targets represented:", int(block_plan["n_targets"].sum()))
display(block_plan.head(20))


Shard 1/1: selected 743 of 1486 blocks.
Targets in selected shard: 7432 of 14774


Blocks selected: 743
Targets represented: 7432


,block_id,HARPNUM,start,end,n_targets,n_positive,global_block_index,num_shards,shard_index
0,2025_HARP12511_20250101_0100_20250101_0412,12511,2025-01-01 01:00:00,2025-01-01 04:12:00,3,0,1,2,1
1,2025_HARP12506_20250101_0836_20250102_0724,12506,2025-01-01 08:36:00,2025-01-02 07:24:00,15,0,3,2,1
2,2025_HARP12511_20250101_2024_20250101_2024,12511,2025-01-01 20:24:00,2025-01-01 20:24:00,1,0,5,2,1
3,2025_HARP12506_20250102_0900_20250103_0724,12506,2025-01-02 09:00:00,2025-01-03 07:24:00,15,0,7,2,1
4,2025_HARP12511_20250102_1248_20250103_1148,12511,2025-01-02 12:48:00,2025-01-03 11:48:00,15,0,9,2,1
5,2025_HARP12506_20250103_0900_20250104_0724,12506,2025-01-03 09:00:00,2025-01-04 07:24:00,15,0,11,2,1
6,2025_HARP12515_20250103_2036_20250104_0612,12515,2025-01-03 20:36:00,2025-01-04 06:12:00,7,0,13,2,1
7,2025_HARP12535_20250104_0800_20250105_0624,12535,2025-01-04 08:00:00,2025-01-05 06:24:00,15,0,15,2,1
8,2025_HARP12515_20250104_0924_20250105_0748,12515,2025-01-04 09:24:00,2025-01-05 07:48:00,15,0,17,2,1
9,2025_HARP12540_20250104_1512_20250105_1336,12540,2025-01-04 15:12:00,2025-01-05 13:36:00,15,0,19,2,1


## 6. Discover completed outputs and restore checkpoints

In [9]:

def download_if_exists(gcp_path, local_path):
    if not gcp_exists(gcp_path):
        return False
    run_command(
        ["gcloud", "storage", "cp", gcp_path, str(local_path)],
        check=True,
    )
    return True


download_if_exists(
    f"{GCP_WORKER_META}/{LOCAL_LOG.name}",
    LOCAL_LOG,
)
download_if_exists(
    f"{GCP_WORKER_META}/{LOCAL_BLOCK_LOG.name}",
    LOCAL_BLOCK_LOG,
)

sample_log = (
    pd.read_csv(LOCAL_LOG, low_memory=False)
    if LOCAL_LOG.exists() and LOCAL_LOG.stat().st_size > 0
    else pd.DataFrame()
)
block_log = (
    pd.read_csv(LOCAL_BLOCK_LOG, low_memory=False)
    if LOCAL_BLOCK_LOG.exists() and LOCAL_BLOCK_LOG.stat().st_size > 0
    else pd.DataFrame()
)

# The object listing is the source of truth for completed model-ready files.
listing = run_command(
    ["gcloud", "storage", "ls", "--recursive", GCP_OUTPUT_ROOT],
    check=False,
)
completed_sample_ids = {
    Path(line.strip()).stem
    for line in listing.stdout.splitlines()
    if line.strip().endswith(".npz")
}

print("Completed GCP samples already present:", len(completed_sample_ids))
print("Sample log rows:", len(sample_log))
print("Block log rows:", len(block_log))


Completed GCP samples already present: 12101
Sample log rows: 6702
Block log rows: 743


## 7. Retry-safe JSOC block export

## v2 cadence-phase fix
This version detects multiple 96-minute cadence phases inside one HARP block, reuses any compatible cached FITS files, and submits extra sequence exports only for uncovered timestamp phases.


In [10]:

def parse_jsoc_time(value):
    try:
        return pd.Timestamp(drms.to_datetime(str(value)))
    except Exception:
        text = str(value).replace("_TAI", "").replace("Z", "")
        return pd.to_datetime(text, errors="coerce")


def extract_request_id(message):
    match = re.search(r"(JSOC_\d{8}_\d+)", str(message))
    return match.group(1) if match else None


def wait_for_existing_request(request_id):
    print("Waiting for existing RequestID:", request_id)
    old_request = jsoc.export_from_id(request_id)
    old_request.wait(
        timeout=JSOC_WAIT_TIMEOUT_SEC,
        sleep=15,
        retries_notfound=30,
    )
    print(
        "Existing request status:",
        old_request.status,
        "succeeded:",
        old_request.has_succeeded(),
    )


def submit_export_retry_safe(query_string, process):
    delay = JSOC_INITIAL_BACKOFF_SEC
    last_error = None

    for attempt in range(1, JSOC_MAX_RETRIES + 1):
        try:
            print(
                f"JSOC export attempt {attempt}/{JSOC_MAX_RETRIES}"
            )
            request = jsoc.export(
                query_string,
                method="url",
                protocol="fits",
                email=JSOC_EMAIL,
                process=process,
            )
            request.wait(
                timeout=JSOC_WAIT_TIMEOUT_SEC,
                sleep=15,
                retries_notfound=30,
            )

            if not request.has_succeeded():
                raise RuntimeError(
                    f"Request failed: id={request.id}, "
                    f"status={request.status}"
                )
            return request

        except DrmsExportError as error:
            last_error = error
            message = str(error)

            if "pending export requests" not in message.lower():
                raise

            request_id = extract_request_id(message)
            print("JSOC pending-request protection triggered.")
            print(message)

            if request_id:
                try:
                    wait_for_existing_request(request_id)
                except Exception as wait_error:
                    print("Could not reopen old request:", repr(wait_error))

            print(f"Sleeping {delay} seconds...")
            time.sleep(delay)
            delay = min(delay * 2, JSOC_MAX_BACKOFF_SEC)

    raise RuntimeError(
        f"JSOC remained busy after all retries: {last_error}"
    )


def format_query_time(timestamp):
    return pd.Timestamp(timestamp).strftime("%Y-%m-%dT%H:%M:%S.000")


def split_block_into_cadence_segments(block):
    """Split a HARP block when target times change cadence phase."""
    block = block.sort_values("T_REC_dt").reset_index(drop=True).copy()
    cadence_seconds = TARGET_CADENCE_MIN * 60
    segments = []
    current_rows = [0]

    for position in range(1, len(block)):
        previous_time = pd.Timestamp(block.loc[position - 1, "T_REC_dt"])
        current_time = pd.Timestamp(block.loc[position, "T_REC_dt"])
        gap_seconds = (current_time - previous_time).total_seconds()
        cadence_steps = max(1, int(round(gap_seconds / cadence_seconds)))
        phase_error_seconds = abs(gap_seconds - cadence_steps * cadence_seconds)

        if phase_error_seconds > MAX_TARGET_TIME_DIFFERENCE_SEC:
            segments.append(block.loc[current_rows].copy())
            current_rows = [position]
        else:
            current_rows.append(position)

    if current_rows:
        segments.append(block.loc[current_rows].copy())

    return [segment.reset_index(drop=True) for segment in segments]


def files_cover_targets(files, target_frame):
    """Check that every target has a FITS file within the time tolerance."""
    files = [Path(item) for item in files if str(item).lower().endswith(".fits")]
    if not files:
        return False

    try:
        indexed = index_downloaded_files(files)
    except Exception:
        return False

    available_times = [item[0] for item in indexed]
    for target in pd.to_datetime(target_frame["T_REC_dt"]):
        nearest_delta = min(
            abs((timestamp - pd.Timestamp(target)).total_seconds())
            for timestamp in available_times
        )
        if nearest_delta > MAX_TARGET_TIME_DIFFERENCE_SEC:
            return False
    return True


def build_segment_export(segment, wavelength, segment_directory):
    segment_directory.mkdir(parents=True, exist_ok=True)
    metadata_json = segment_directory / "export_metadata.json"

    for partial in segment_directory.glob("*.part"):
        partial.unlink(missing_ok=True)

    existing_fits = sorted(segment_directory.glob("*.fits"))
    if metadata_json.exists() and existing_fits and files_cover_targets(existing_fits, segment):
        with metadata_json.open() as handle:
            saved = json.load(handle)
        print(f"♻️ Reusing {len(existing_fits)} cadence-aligned files for {wavelength} Å")
        return existing_fits, saved

    segment = segment.sort_values("T_REC_dt").reset_index(drop=True)
    start = pd.Timestamp(segment["T_REC_dt"].min())
    end = pd.Timestamp(segment["T_REC_dt"].max())
    duration_minutes = max(
        TARGET_CADENCE_MIN,
        int(math.ceil((end - start).total_seconds() / 60.0)) + TARGET_CADENCE_MIN,
    )

    reference_time = start + (end - start) / 2
    reference_index = (segment["T_REC_dt"] - reference_time).abs().idxmin()
    reference_row = segment.loc[reference_index]
    reference_geometry = target_geometry(reference_row)

    max_target_box = max(target_geometry(row)["box_arcsec"] for _, row in segment.iterrows())
    patch_size = np.clip(
        max_target_box + BLOCK_PATCH_MARGIN_ARCSEC,
        MIN_BLOCK_PATCH_ARCSEC,
        MAX_BLOCK_PATCH_ARCSEC,
    )

    query_string = (
        f"aia.lev1_euv_12s"
        f"[{format_query_time(start)}/{duration_minutes}m@{TARGET_CADENCE_MIN}m]"
        f"[{int(wavelength)}]"
        f"{{image}}"
    )

    process = {
        "im_patch": {
            "t_ref": format_query_time(reference_time),
            "t": 0,
            "r": 0,
            "c": 0,
            "locunits": "arcsec",
            "boxunits": "arcsec",
            "x": reference_geometry["x_arcsec"],
            "y": reference_geometry["y_arcsec"],
            "width": float(patch_size),
            "height": float(patch_size),
        }
    }

    print("Segment query:", query_string)
    print("Segment reference:", reference_time, "| targets:", len(segment), "| patch arcsec:", float(patch_size))

    request = submit_export_retry_safe(query_string, process)
    request.download(segment_directory, timeout=600)

    fits_files = sorted(segment_directory.glob("*.fits"))
    if not fits_files:
        raise FileNotFoundError(f"No FITS files downloaded for {wavelength} Å segment.")
    if not files_cover_targets(fits_files, segment):
        raise RuntimeError(
            f"Downloaded {wavelength} Å segment does not cover all target timestamps within "
            f"{MAX_TARGET_TIME_DIFFERENCE_SEC} seconds."
        )

    metadata = {
        "request_id": request.id,
        "query": query_string,
        "wavelength": int(wavelength),
        "reference_time": str(reference_time),
        "segment_start": str(start),
        "segment_end": str(end),
        "segment_targets": int(len(segment)),
        "patch_size_arcsec": float(patch_size),
        "reference_geometry": reference_geometry,
        "n_files": len(fits_files),
        "email": JSOC_EMAIL,
    }
    with metadata_json.open("w") as handle:
        json.dump(metadata, handle, indent=2)

    time.sleep(JSOC_COOLDOWN_SEC)
    return fits_files, metadata


def build_block_export(block, wavelength, block_directory):
    """Export one or more cadence-aligned sequences for a HARP block."""
    block_directory.mkdir(parents=True, exist_ok=True)
    wave_directory = block_directory / str(wavelength)
    wave_directory.mkdir(parents=True, exist_ok=True)

    segments = split_block_into_cadence_segments(block)
    print(
        f"{wavelength} Å cadence segments:",
        len(segments),
        [(str(s["T_REC_dt"].min()), str(s["T_REC_dt"].max()), len(s)) for s in segments],
    )

    request_ids = []
    for segment_number, segment in enumerate(segments, start=1):
        cached_files = sorted(wave_directory.rglob("*.fits"))
        if files_cover_targets(cached_files, segment):
            print(
                f"♻️ Segment {segment_number}/{len(segments)} already covered by cached "
                f"{wavelength} Å files."
            )
            continue

        start = pd.Timestamp(segment["T_REC_dt"].min())
        end = pd.Timestamp(segment["T_REC_dt"].max())
        segment_name = (
            f"segment_{segment_number:02d}_"
            f"{start.strftime('%Y%m%d_%H%M')}_"
            f"{end.strftime('%Y%m%d_%H%M')}"
        )
        segment_directory = wave_directory / segment_name
        _, segment_metadata = build_segment_export(segment, wavelength, segment_directory)
        request_ids.append(str(segment_metadata["request_id"]))

    all_fits = sorted(wave_directory.rglob("*.fits"))
    if not all_fits:
        raise FileNotFoundError(f"No complete FITS files available for {wavelength} Å.")

    if not files_cover_targets(all_fits, block):
        uncovered = []
        indexed = index_downloaded_files(all_fits)
        available_times = [item[0] for item in indexed]
        for target in pd.to_datetime(block["T_REC_dt"]):
            nearest_delta = min(
                abs((timestamp - pd.Timestamp(target)).total_seconds())
                for timestamp in available_times
            )
            if nearest_delta > MAX_TARGET_TIME_DIFFERENCE_SEC:
                uncovered.append({
                    "target": str(target),
                    "nearest_delta_seconds": float(nearest_delta),
                })
        raise RuntimeError(f"{wavelength} Å block remains incompletely covered: {uncovered[:10]}")

    for metadata_path in wave_directory.rglob("export_metadata.json"):
        try:
            with metadata_path.open() as handle:
                item = json.load(handle)
            request_id = item.get("request_id")
            if request_id:
                request_ids.append(str(request_id))
        except Exception:
            pass

    request_ids = sorted(set(request_ids))
    combined_metadata = {
        "request_id": ",".join(request_ids) if request_ids else "cached",
        "request_ids": request_ids,
        "wavelength": int(wavelength),
        "n_segments": int(len(segments)),
        "n_files": int(len(all_fits)),
        "coverage_verified": True,
        "max_time_difference_seconds": int(MAX_TARGET_TIME_DIFFERENCE_SEC),
        "email": JSOC_EMAIL,
    }
    return all_fits, combined_metadata


def fits_observation_time(path):
    with fits.open(path, memmap=False) as hdul:
        headers = [
            hdu.header
            for hdu in hdul
            if getattr(hdu, "header", None) is not None
        ]

    for header in headers:
        for key in ["T_REC", "DATE-OBS", "DATE_OBS", "T_OBS"]:
            if key in header:
                parsed = parse_jsoc_time(header[key])
                if not pd.isna(parsed):
                    return pd.Timestamp(parsed)

    raise ValueError(f"No observation time found in {path}")


def index_downloaded_files(files):
    indexed = []
    for path in files:
        try:
            timestamp = fits_observation_time(path)
            indexed.append((timestamp, path))
        except Exception as error:
            print("Skipping unreadable FITS time:", path, repr(error))

    if not indexed:
        raise RuntimeError("No downloaded FITS file has a valid timestamp.")

    return sorted(indexed, key=lambda item: item[0])


def nearest_file(indexed_files, target_time):
    target_time = pd.Timestamp(target_time)
    timestamp, path = min(
        indexed_files,
        key=lambda item: abs((item[0] - target_time).total_seconds()),
    )
    difference = abs((timestamp - target_time).total_seconds())

    if difference > MAX_TARGET_TIME_DIFFERENCE_SEC:
        raise ValueError(
            f"Nearest AIA file is {difference:.1f}s from target "
            f"{target_time}."
        )

    return path, timestamp, float(difference)


## 8. Save, upload and checkpoint model-ready samples

In [11]:

def free_disk_gb(path):
    usage = shutil.disk_usage(path)
    return usage.free / (1024 ** 3)


def upload_verified(local_path, gcp_path):
    run_command(
        ["gcloud", "storage", "cp", str(local_path), gcp_path],
        check=True,
    )
    if not gcp_exists(gcp_path):
        raise RuntimeError(f"Upload verification failed: {gcp_path}")


def append_checkpoint(frame, row, local_path, gcp_path):
    updated = pd.concat(
        [frame, pd.DataFrame([row])],
        ignore_index=True,
    )
    updated = updated.drop_duplicates(
        subset=["sample_id"],
        keep="last",
    )
    updated.to_csv(local_path, index=False)
    run_command(
        ["gcloud", "storage", "cp", str(local_path), gcp_path],
        check=True,
    )
    return updated


def append_block_checkpoint(frame, row):
    updated = pd.concat(
        [frame, pd.DataFrame([row])],
        ignore_index=True,
    )
    updated = updated.drop_duplicates(
        subset=["block_id"],
        keep="last",
    )
    updated.to_csv(LOCAL_BLOCK_LOG, index=False)
    run_command(
        [
            "gcloud", "storage", "cp",
            str(LOCAL_BLOCK_LOG),
            f"{GCP_WORKER_META}/{LOCAL_BLOCK_LOG.name}",
        ],
        check=True,
    )
    return updated


def process_block(block_id, block, sample_log):
    block_started = time.time()
    block_directory = LOCAL_TEMP / block_id
    block_directory.mkdir(parents=True, exist_ok=True)

    pending = block[
        ~block["sample_id"].isin(completed_sample_ids)
    ].copy()

    if len(pending) == 0:
        return sample_log, {
            "block_id": block_id,
            "status": "already_complete",
            "n_targets": len(block),
            "n_saved_this_run": 0,
            "elapsed_minutes": 0.0,
            "message": "all_samples_already_in_gcp",
        }

    if free_disk_gb(LOCAL_ROOT) < MIN_FREE_DISK_GB:
        raise RuntimeError(
            f"Free disk below {MIN_FREE_DISK_GB} GB."
        )

    wavelength_indices = {}
    wavelength_export_meta = {}

    for wavelength in AIA_WAVELENGTHS:
        print("\n" + "-" * 70)
        print(block_id, "| wavelength", wavelength)
        files, export_meta = build_block_export(
            block,
            wavelength,
            block_directory,
        )
        wavelength_indices[wavelength] = index_downloaded_files(files)
        wavelength_export_meta[wavelength] = export_meta

    saved_this_block = 0

    for _, row in pending.iterrows():
        sample_id = str(row["sample_id"])
        target_time = pd.Timestamp(row["T_REC_dt"])
        channels = []
        channel_meta = {}

        try:
            for wavelength in AIA_WAVELENGTHS:
                path, used_time, delta_seconds = nearest_file(
                    wavelength_indices[wavelength],
                    target_time,
                )
                channel, crop_meta = crop_target_from_block(path, row)
                channels.append(channel)

                channel_meta[str(wavelength)] = {
                    "source_file": path.name,
                    "used_time": str(used_time),
                    "delta_seconds": delta_seconds,
                    "request_id": wavelength_export_meta[wavelength][
                        "request_id"
                    ],
                    "crop": crop_meta,
                }

            tensor = np.stack(channels, axis=-1).astype(np.float32)

            if tensor.shape != (IMAGE_SIZE, IMAGE_SIZE, 6):
                raise ValueError(f"Unexpected shape: {tensor.shape}")
            if not np.isfinite(tensor).all():
                raise ValueError("Tensor contains NaN or infinity.")

            year_directory = LOCAL_OUTPUT / str(TARGET_YEAR)
            year_directory.mkdir(parents=True, exist_ok=True)
            local_npz = year_directory / f"{sample_id}.npz"

            np.savez_compressed(
                local_npz,
                x=tensor,
                y=np.array(
                    int(row["label_48h_final"]),
                    dtype=np.int64,
                ),
                sample_id=np.array(sample_id),
                T_REC_dt=np.array(str(target_time)),
                HARPNUM=np.array(int(row["HARPNUM"]), dtype=np.int64),
                NOAA_AR_clean=np.array(
                    int(row["NOAA_AR_clean"]),
                    dtype=np.int64,
                ),
                label_48h_final=np.array(
                    int(row["label_48h_final"]),
                    dtype=np.int64,
                ),
                wavelengths=np.array(
                    AIA_WAVELENGTHS,
                    dtype=np.int64,
                ),
                source=np.array(
                    "JSOC HARP-block tracked im_patch + local WCS crop"
                ),
                block_id=np.array(block_id),
                channel_metadata=np.array(json.dumps(channel_meta)),
            )

            gcp_npz = f"{GCP_OUTPUT_ROOT}/{local_npz.name}"
            upload_verified(local_npz, gcp_npz)
            completed_sample_ids.add(sample_id)
            saved_this_block += 1

            sample_row = {
                "sample_id": sample_id,
                "block_id": block_id,
                "T_REC_dt": str(target_time),
                "HARPNUM": int(row["HARPNUM"]),
                "NOAA_AR_clean": int(row["NOAA_AR_clean"]),
                "label_48h_final": int(row["label_48h_final"]),
                "status": "saved",
                "shape": str(tensor.shape),
                "gcp_path": gcp_npz,
                "message": "success",
                "updated_at_utc": pd.Timestamp.utcnow().isoformat(),
            }

            sample_log = append_checkpoint(
                sample_log,
                sample_row,
                LOCAL_LOG,
                f"{GCP_WORKER_META}/{LOCAL_LOG.name}",
            )

            local_npz.unlink(missing_ok=True)
            print("✅", sample_id)

        except Exception as error:
            sample_row = {
                "sample_id": sample_id,
                "block_id": block_id,
                "T_REC_dt": str(target_time),
                "HARPNUM": int(row["HARPNUM"]),
                "NOAA_AR_clean": int(row["NOAA_AR_clean"]),
                "label_48h_final": int(row["label_48h_final"]),
                "status": "error",
                "shape": None,
                "gcp_path": None,
                "message": repr(error),
                "updated_at_utc": pd.Timestamp.utcnow().isoformat(),
            }

            sample_log = append_checkpoint(
                sample_log,
                sample_row,
                LOCAL_LOG,
                f"{GCP_WORKER_META}/{LOCAL_LOG.name}",
            )
            print("❌", sample_id, repr(error))

    # The whole block is retained only when a target failed, allowing reuse.
    current_errors = sample_log[
        (sample_log["block_id"] == block_id)
        & (sample_log["status"] == "error")
    ] if len(sample_log) else pd.DataFrame()

    if len(current_errors) == 0:
        shutil.rmtree(block_directory, ignore_errors=True)

    elapsed_minutes = (time.time() - block_started) / 60.0

    return sample_log, {
        "block_id": block_id,
        "status": "completed",
        "n_targets": len(block),
        "n_pending_at_start": len(pending),
        "n_saved_this_run": saved_this_block,
        "elapsed_minutes": round(elapsed_minutes, 3),
        "message": (
            "success"
            if len(current_errors) == 0
            else f"{len(current_errors)} sample errors retained for retry"
        ),
    }


## 9. Execute selected blocks

In [12]:

blocks_to_run = block_plan.copy()

if MAX_BLOCKS_THIS_RUN is not None:
    blocks_to_run = blocks_to_run.head(MAX_BLOCKS_THIS_RUN)

print("Blocks this run:", len(blocks_to_run))

for position, plan_row in blocks_to_run.iterrows():
    block_id = plan_row["block_id"]
    block = block_frames[block_id]

    print("\n" + "=" * 90)
    print(
        f"BLOCK {position + 1}/{len(blocks_to_run)} | "
        f"{block_id} | targets={len(block)}"
    )
    print("=" * 90)

    try:
        sample_log, block_result = process_block(
            block_id,
            block,
            sample_log,
        )
    except Exception as error:
        block_result = {
            "block_id": block_id,
            "status": "error",
            "n_targets": len(block),
            "n_pending_at_start": None,
            "n_saved_this_run": 0,
            "elapsed_minutes": None,
            "message": repr(error),
        }
        print("BLOCK ERROR:", repr(error))

    block_log = append_block_checkpoint(
        block_log,
        block_result,
    )
    display(pd.DataFrame([block_result]))

print("\nRun finished.")
print(
    "Completed model-ready objects now visible in GCP:",
    len(completed_sample_ids),
)


Blocks this run: 743

BLOCK 1/743 | 2025_HARP12511_20250101_0100_20250101_0412 | targets=3


,block_id,status,n_targets,n_saved_this_run,elapsed_minutes,message
0,2025_HARP12511_20250101_0100_20250101_0412,already_complete,3,0,0.0,all_samples_already_in_gcp



BLOCK 2/743 | 2025_HARP12506_20250101_0836_20250102_0724 | targets=15


,block_id,status,n_targets,n_saved_this_run,elapsed_minutes,message
0,2025_HARP12506_20250101_0836_20250102_0724,already_complete,15,0,0.0,all_samples_already_in_gcp



BLOCK 3/743 | 2025_HARP12511_20250101_2024_20250101_2024 | targets=1


...


BLOCK 742/743 | 2025_HARP14215_20251231_1936_20251231_2248 | targets=3


,block_id,status,n_targets,n_saved_this_run,elapsed_minutes,message
0,2025_HARP14215_20251231_1936_20251231_2248,already_complete,3,0,0.0,all_samples_already_in_gcp



BLOCK 743/743 | 2025_HARP14239_20251231_2324_20251231_2324 | targets=1


,block_id,status,n_targets,n_saved_this_run,elapsed_minutes,message
0,2025_HARP14239_20251231_2324_20251231_2324,already_complete,1,0,0.0,all_samples_already_in_gcp



Run finished.
Completed model-ready objects now visible in GCP: 12119


## 10. Audit expected versus completed

In [13]:

fresh_listing = run_command(
    ["gcloud", "storage", "ls", "--recursive", GCP_OUTPUT_ROOT],
    check=False,
)
actual_ids = {
    Path(line.strip()).stem
    for line in fresh_listing.stdout.splitlines()
    if line.strip().endswith(".npz")
}

expected_ids = set(df["sample_id"].astype(str))

if RUN_MODE == "BLOCK_CANARY" or (
    RUN_MODE == "PRODUCTION" and NUM_SHARDS > 1
):
    selected_block_ids = set(block_plan["block_id"])
    expected_ids = set(
        pd.concat(
            [block_frames[item] for item in selected_block_ids],
            ignore_index=True,
        )["sample_id"].astype(str)
    )

missing_ids = expected_ids - actual_ids
unexpected_ids = actual_ids - set(df["sample_id"].astype(str))

print("Expected in this run scope:", len(expected_ids))
print("Completed in GCP:", len(actual_ids.intersection(expected_ids)))
print("Missing:", len(missing_ids))
print("Unexpected:", len(unexpected_ids))

audit = pd.DataFrame(
    {
        "metric": [
            "expected_scope",
            "completed_scope",
            "missing_scope",
            "unexpected_year_objects",
        ],
        "value": [
            len(expected_ids),
            len(actual_ids.intersection(expected_ids)),
            len(missing_ids),
            len(unexpected_ids),
        ],
    }
)
display(audit)

missing_path = LOCAL_META / f"missing_ids_{WORKER_ID}.txt"
missing_path.write_text("\n".join(sorted(missing_ids)))
run_command(
    [
        "gcloud", "storage", "cp",
        str(missing_path),
        f"{GCP_WORKER_META}/{missing_path.name}",
    ],
    check=True,
)


Expected in this run scope: 7432
Completed in GCP: 6966
Missing: 466
Unexpected: 0


,metric,value
0,expected_scope,7432
1,completed_scope,6966
2,missing_scope,466
3,unexpected_year_objects,0


CompletedProcess(args=['gcloud', 'storage', 'cp', '/home/abmoses2000/solar_flare_aia/harp_block_miner/production_aia2025-s1/metadata/missing_ids_aia2025-s1.txt', 'gs://suryabench-sharp-pipeline-bamidele/jsoc_2025_2026_production_v1/metadata/workers/aia2025-s1/missing_ids_aia2025-s1.txt'], returncode=0, stdout='', stderr='Copying file:///home/abmoses2000/solar_flare_aia/harp_block_miner/production_aia2025-s1/metadata/missing_ids_aia2025-s1.txt to gs://suryabench-sharp-pipeline-bamidele/jsoc_2025_2026_production_v1/metadata/workers/aia2025-s1/missing_ids_aia2025-s1.txt\n  \n.\n')

## 11. Block-canary comparison with individual pilot outputs

In [14]:

if RUN_MODE != "BLOCK_CANARY":
    print("Comparison is only used in BLOCK_CANARY mode.")
else:
    comparison_root = LOCAL_ROOT / "comparison"
    comparison_root.mkdir(parents=True, exist_ok=True)

    comparison_rows = []

    for sample_id in CANARY_SAMPLE_IDS[TARGET_YEAR]:
        block_path = comparison_root / f"block_{sample_id}.npz"
        pilot_path = comparison_root / f"pilot_{sample_id}.npz"

        block_gcp = f"{GCP_OUTPUT_ROOT}/{sample_id}.npz"
        pilot_gcp = f"{PILOT_GCP_ROOT}/{sample_id}.npz"

        if not gcp_exists(block_gcp) or not gcp_exists(pilot_gcp):
            print("Comparison unavailable:", sample_id)
            continue

        run_command(
            ["gcloud", "storage", "cp", block_gcp, str(block_path)]
        )
        run_command(
            ["gcloud", "storage", "cp", pilot_gcp, str(pilot_path)]
        )

        with np.load(block_path, allow_pickle=True) as block_npz:
            block_x = block_npz["x"]
        with np.load(pilot_path, allow_pickle=True) as pilot_npz:
            pilot_x = pilot_npz["x"]

        for channel_index, wavelength in enumerate(AIA_WAVELENGTHS):
            first = block_x[:, :, channel_index]
            second = pilot_x[:, :, channel_index]

            correlation = float(
                np.corrcoef(first.ravel(), second.ravel())[0, 1]
            )
            ssim = float(
                structural_similarity(
                    first,
                    second,
                    data_range=1.0,
                )
            )

            comparison_rows.append(
                {
                    "sample_id": sample_id,
                    "wavelength": wavelength,
                    "pearson_r": correlation,
                    "ssim": ssim,
                }
            )

        fig, axes = plt.subplots(2, 6, figsize=(18, 6))
        for channel_index, wavelength in enumerate(AIA_WAVELENGTHS):
            axes[0, channel_index].imshow(
                pilot_x[:, :, channel_index],
                origin="lower",
                cmap="gray",
            )
            axes[0, channel_index].set_title(f"Pilot {wavelength} Å")
            axes[0, channel_index].axis("off")

            axes[1, channel_index].imshow(
                block_x[:, :, channel_index],
                origin="lower",
                cmap="gray",
            )
            axes[1, channel_index].set_title(f"Block {wavelength} Å")
            axes[1, channel_index].axis("off")

        fig.suptitle(sample_id)
        plt.tight_layout()
        plt.show()

    comparison_df = pd.DataFrame(comparison_rows)
    display(comparison_df)

    if len(comparison_df):
        print("\nMean correlation:", comparison_df["pearson_r"].mean())
        print("Mean SSIM:", comparison_df["ssim"].mean())

        comparison_path = (
            LOCAL_META / f"block_vs_pilot_{WORKER_ID}.csv"
        )
        comparison_df.to_csv(comparison_path, index=False)
        run_command(
            [
                "gcloud", "storage", "cp",
                str(comparison_path),
                f"{GCP_WORKER_META}/{comparison_path.name}",
            ],
            check=True,
        )


Comparison is only used in BLOCK_CANARY mode.



## 12. Acceptance gate

Before switching to `PRODUCTION`, confirm:

1. all target timestamps in the selected block are represented;
2. six AIA channels exist for every saved sample;
3. output shape is `(512, 512, 6)`;
4. all values are finite and within `[0, 1]`;
5. AIA-to-SHARP time differences are no more than 180 seconds;
6. active regions are centred and not clipped;
7. block-generated images visually match the individual pilot images;
8. correlation and SSIM are scientifically acceptable;
9. no unexpected sample IDs are present;
10. block runtime is materially faster than the former 7–8 minutes per sample.

## Starting production on the VM

Once the block canary passes, place the notebook in:

```text
~/solar_flare_aia/notebooks/
```

Then run the 2025 worker:

```bash
tmux new -s aia2025
source ~/solar_flare_aia/venv/bin/activate
export TARGET_YEAR=2025
export JSOC_EMAIL=abmoses2000@gmail.com
export WORKER_ID=aia2025
export RUN_MODE=PRODUCTION
jupyter nbconvert \
  --to notebook \
  --execute ~/solar_flare_aia/notebooks/05_AIA_JSOC_HARP_BLOCK_MINER_VM_READY.ipynb \
  --ExecutePreprocessor.timeout=-1 \
  --output ~/solar_flare_aia/logs/aia2025_executed.ipynb
```

Detach from `tmux` with `Ctrl+B`, then `D`.

A second worker can process 2026 using `worky4work@gmail.com`, but first verify that two simultaneous block workers do not overload the VM or JSOC.
